# DAPO (Distributional Advantage Policy Optimization)

考虑优势函数分布的策略优化方法

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [ ]:
class DAPOTrainer:
    def __init__(self, num_quantiles=51, risk_sensitivity=0.0):
        self.num_quantiles = num_quantiles
        self.risk_sensitivity = risk_sensitivity
    
    def compute_risk_adjusted_advantage(self, quantile_advantages):
        mean_adv = np.mean(quantile_advantages)
        var_adv = np.var(quantile_advantages)
        return mean_adv + self.risk_sensitivity * np.sqrt(var_adv)
    
    def dapo_loss(self, log_probs, quantile_advantages):
        batch_size = len(log_probs)
        risk_adjusted_advantages = np.zeros(batch_size)
        
        for i in range(batch_size):
            risk_adjusted_advantages[i] = self.compute_risk_adjusted_advantage(
                quantile_advantages[i]
            )
        
        policy_loss = -np.mean(log_probs * risk_adjusted_advantages)
        
        metrics = {
            'loss': policy_loss,
            'advantage_mean': np.mean(risk_adjusted_advantages)
        }
        
        return policy_loss, metrics

In [ ]:
# 示例
dapo = DAPOTrainer(num_quantiles=51, risk_sensitivity=0.0)

batch_size = 4
num_quantiles = 51

log_probs = np.random.randn(batch_size) * 0.5 - 2.0
quantile_advantages = np.random.randn(batch_size, num_quantiles) * 0.3

for i in range(batch_size):
    quantile_advantages[i] = np.sort(quantile_advantages[i])

loss, metrics = dapo.dapo_loss(log_probs, quantile_advantages)
print(f"损失: {loss:.4f}")
print(f"指标: {metrics}")

In [ ]:
# 可视化分位数分布
sample_quantiles = quantile_advantages[0]
quantile_positions = np.linspace(0, 1, num_quantiles)

plt.figure(figsize=(10, 6))
plt.plot(quantile_positions, sample_quantiles, 'o-')
plt.axhline(y=np.mean(sample_quantiles), color='r', linestyle='--', label='均值')
plt.axhline(y=sample_quantiles[25], color='g', linestyle='--', label='中位数')
plt.xlabel('分位数')
plt.ylabel('优势值')
plt.title('优势函数的分位数分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()